In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py 
import sys 
import os, glob

# Path to repo root (two directories above notebook)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)

from utils.backtracking import get_charge_event_hits, hit_backtracker, get_ancestry # Now this works

plt.style.use('../utils/dune.mplstyle')

import time 
from collections import defaultdict

In [2]:
file='/global/cfs/cdirs/dune/users/lmlepin/2x2_neutron_prod/AmBe_top_mod2_PROD_03-21/FLOW/2x2_QGSP_BERT_HP_AmBe_1774119367_0_TIME_MOD.FLOW.hdf5'
h5_file = h5py.File(file,'r') 
print(f"This file keys: {h5_file.keys()}")

This file keys: <KeysViewHDF5 ['charge', 'combined', 'geometry_info', 'lar_info', 'light', 'mc_truth', 'run_info']>


In [3]:
trajectories = h5_file['mc_truth/trajectories/data']
segments = h5_file['mc_truth/segments/data']
vertex = h5_file['mc_truth/interactions/data']

charge_prompt_hits = h5_file['charge/calib_prompt_hits/data']
charge_prompt_hits_bt = h5_file['mc_truth/calib_prompt_hit_backtrack/data']

print(f"Trajectories keys: {trajectories.dtype.names}")
print(f"Segments keys: {segments.dtype.names}")
print(f"Charge hits keys {charge_prompt_hits.dtype.names}")
print(f"Charge hits bt keys {charge_prompt_hits_bt.dtype.names}")

Trajectories keys: ('event_id', 'vertex_id', 'traj_id', 'file_traj_id', 'parent_id', 'primary', 'E_start', 'pxyz_start', 'xyz_start', 't_start', 'E_end', 'pxyz_end', 'xyz_end', 't_end', 'pdg_id', 'start_process', 'start_subprocess', 'end_process', 'end_subprocess', 'dist_travel')
Segments keys: ('event_id', 'vertex_id', 'segment_id', 'z_end', 'traj_id', 'file_traj_id', 'tran_diff', 'z_start', 'x_end', 'y_end', 'n_electrons', 'pdg_id', 'x_start', 'y_start', 't_start', 't0_start', 't0_end', 't0', 'dx', 'long_diff', 'pixel_plane', 't_end', 'dEdx', 'dE', 't', 'y', 'x', 'z', 'n_photons')
Charge hits keys ('id', 'x', 'y', 'z', 't_drift', 'ts_pps', 'io_group', 'io_channel', 'chip_id', 'channel_id', 'Q_raw', 'Q', 'E', 'is_disabled')
Charge hits bt keys ('event_ids', 'segment_ids', 'fraction', 'file_traj_ids', 'fraction_traj')


In [15]:
n_events_minus = 0
n_minus_hits_per_ev = [] 
for i in range(len(vertex)):
    #if(i>3):
    #    break
    phits, phits_bt = get_charge_event_hits(i,h5_file)
    this_ev_id = vertex['event_id'][i]
    this_ev_traj = trajectories[trajectories['event_id']==this_ev_id]
    this_ev_segs = segments[segments['event_id']==this_ev_id]
    if(len(phits)!=len(phits_bt)):
        print(f"[WARNING]: On this entry {i}, the number of bt hits doesnt match the number of hits")

    mc_hits = hit_backtracker(phits,phits_bt,this_ev_segs,this_ev_traj)
    if(i==0):
        print(mc_hits.dtype.names)
    print(mc_hits)
    print("---------")
    if(-1 in mc_hits['best_segment_id']):
        n_events_minus+=1
        n_minus_hits_per_ev.append(len(mc_hits[mc_hits['best_segment_id']==-1]))

print(f"Number of events with hits w/o segment_id {n_events_minus}")
print(f"Bin count of hits w/o segment_id: {np.bincount(n_minus_hits_per_ev)}")

('id', 'x', 'y', 'z', 't_drift', 'ts_pps', 'io_group', 'io_channel', 'chip_id', 'channel_id', 'Q_raw', 'Q', 'E', 'is_disabled', 'best_segment_id', 'neutron_process', 'parent_neutron_id')
[(0, 25.91513022, -17.07089996, 64.31629944, 1441., 2001439, 2, 18, 51, 17, 18.7643003, 18.7643003, 0.63152476, False, 1990, 1, 0)]
---------
[(1, -54.47910825, -7.56551266, 61.62818909, 602., 4000600, 6, 19, 81, 47, 10.54156804, 10.54156804, 0.35478335, False, 2880, 1, 0)
 (2, -54.62278898, -7.17753744, 61.62818909, 593., 4000591, 6, 19, 81, 43, 13.14848941, 13.14848941, 0.44252099, False, 2880, 1, 0)
 (3, -54.17578228, -7.17753744, 61.62818909, 621., 4000619, 6, 19, 81, 43, 21.62190756, 21.62190756, 0.7276994 , False, 2878, 1, 0)]
---------
[(4, 44.19795427, -0.2217    , -37.26890182, 1246., 6001244, 3, 20, 109, 34, 26.00667685, 26.00667685, 0.87527167, False, 3587, 0, 1)
 (5, 28.62909944,  8.64630032,  -7.1177001 , 1611., 6001609, 4,  9,  32, 32,  8.46910439,  8.46910439, 0.28503323, False, 3606, 1,